# 01 - Data Cleaning & Preparation

In this first notebook I load the raw sales data, check its quality and prepare it for analysis.

**Steps:**
1. First look at the data
2. Fix data types
3. Missing values and duplicates
4. Consistency checks (do the numbers add up?)
5. Categorical values and outliers
6. Create new columns for the analysis
7. Save the clean data (CSV + SQLite database for the SQL notebook)

**About the data:** sales of a fictional office supplies / tech retailer in the US, 2023-2024.
Each row is one **line of an order** (one product inside an order), so an order can have more than one row.

In [1]:
import pandas as pd
import numpy as np
import sqlite3

# show all columns when printing a dataframe
pd.set_option("display.max_columns", None)

## 1. First look at the data

In [2]:
df = pd.read_csv("../data/raw/sales.csv")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()

Rows: 80,983
Columns: 27


,order_id,order_date,ship_date,deliver_date,ship_mode,order_status,customer_id,customer_name,segment,city,state,region,product_id,product_name,category,sub_category,unit_price,cost_price,quantity,discount,sales,profit,profit_margin,payment_method,days_to_ship,days_to_deliver,is_returned
0,ORD-000001,2023-01-01,2023-01-02,2023-01-04,First Class,Delivered,CUST-05329,Mrs. Dawn Hawkins,Consumer,Lake Angela,Georgia,South,PROD-015,Compact Writing Desk,Furniture,Desks,199,80,1,0.0,199.0,119.0,59.80,Credit Card,1,3,0
1,ORD-000002,2023-01-01,2023-01-02,2023-01-05,Second Class,Delivered,CUST-01501,Jacob Duran,Consumer,Andersonland,Virginia,South,PROD-021,Sticky Notes 12-Pack,Office Supplies,Paper,8,2,1,0.1,7.2,5.2,72.22,Credit Card,1,4,0
2,ORD-000003,2023-01-01,2023-01-02,2023-01-09,Standard Class,Delivered,CUST-02180,Christine Williams,Corporate,Port Amanda,Virginia,South,PROD-023,Highlighter Set 6 Colors,Office Supplies,Writing,7,2,1,0.0,7.0,5.0,71.43,Credit Card,1,8,0
3,ORD-000004,2023-01-01,2023-01-02,2023-01-09,Standard Class,Delivered,CUST-01621,Karen Ortiz,Consumer,Hayesview,North Carolina,South,PROD-003,Google Pixel 8,Technology,Phones,699,490,1,0.0,699.0,209.0,29.90,Credit Card,1,8,0
4,ORD-000005,2023-01-01,2023-01-02,2023-01-07,Standard Class,Delivered,CUST-04672,Robert Hall,Consumer,Delgadofort,Washington,West,PROD-002,Samsung Galaxy S24,Technology,Phones,799,580,1,0.0,799.0,219.0,27.41,Credit Card,1,6,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80983 entries, 0 to 80982
Data columns (total 27 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         80983 non-null  object 
 1   order_date       80983 non-null  object 
 2   ship_date        80983 non-null  object 
 3   deliver_date     80983 non-null  object 
 4   ship_mode        80983 non-null  object 
 5   order_status     80983 non-null  object 
 6   customer_id      80983 non-null  object 
 7   customer_name    80983 non-null  object 
 8   segment          80983 non-null  object 
 9   city             80983 non-null  object 
 10  state            80983 non-null  object 
 11  region           80983 non-null  object 
 12  product_id       80983 non-null  object 
 13  product_name     80983 non-null  object 
 14  category         80983 non-null  object 
 15  sub_category     80983 non-null  object 
 16  unit_price       80983 non-null  int64  
 17  cost_price  

What the columns mean:

| Column | Description |
|---|---|
| `order_id`, `order_date`, `ship_date`, `deliver_date` | order identifier and dates |
| `ship_mode`, `order_status` | shipping option chosen and if the item was delivered or returned |
| `customer_id`, `customer_name`, `segment` | who bought (Consumer, Corporate or Home Office) |
| `city`, `state`, `region` | where the customer is |
| `product_id`, `product_name`, `category`, `sub_category` | what was bought |
| `unit_price`, `cost_price`, `quantity`, `discount` | price paid, cost for the company, units and discount (0.1 = 10%) |
| `sales`, `profit`, `profit_margin` | revenue of the line, profit of the line, and margin in % |
| `payment_method`, `days_to_ship`, `days_to_deliver`, `is_returned` | payment, logistics and return flag (1 = returned) |

The 3 date columns were read as text (`object`), I will fix that next.

## 2. Fix data types

In [4]:
date_columns = ["order_date", "ship_date", "deliver_date"]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], format="%Y-%m-%d")

df[date_columns].dtypes

order_date      datetime64[ns]
ship_date       datetime64[ns]
deliver_date    datetime64[ns]
dtype: object

In [5]:
# what period does the data cover?
print("First order:", df["order_date"].min().date())
print("Last order: ", df["order_date"].max().date())

First order: 2023-01-01
Last order:  2024-12-31


Two full years of data (2023 and 2024), which is good to compare one year against the other.

## 3. Missing values and duplicates

In [6]:
df.isna().sum()

order_id           0
order_date         0
ship_date          0
deliver_date       0
ship_mode          0
order_status       0
customer_id        0
customer_name      0
segment            0
city               0
state              0
region             0
product_id         0
product_name       0
category           0
sub_category       0
unit_price         0
cost_price         0
quantity           0
discount           0
sales              0
profit             0
profit_margin      0
payment_method     0
days_to_ship       0
days_to_deliver    0
is_returned        0
dtype: int64

No missing values in any column. Now the duplicates:

In [7]:
n_duplicates = df.duplicated().sum()
print("Rows that are an exact copy of another row:", n_duplicates)

# look at a few of them (keep=False shows both the original and the copy)
df[df.duplicated(keep=False)].sort_values("order_id").head(6)

Rows that are an exact copy of another row: 68


,order_id,order_date,ship_date,deliver_date,ship_mode,order_status,customer_id,customer_name,segment,city,state,region,product_id,product_name,category,sub_category,unit_price,cost_price,quantity,discount,sales,profit,profit_margin,payment_method,days_to_ship,days_to_deliver,is_returned
1722,ORD-001286,2023-01-26,2023-01-27,2023-01-30,Second Class,Delivered,CUST-02158,Pam Baker,Consumer,New James,Florida,South,PROD-025,3-Ring Binder Set,Office Supplies,Organization,15,5,1,0.0,15.0,10.0,66.67,Bank Transfer,1,4,0
1723,ORD-001286,2023-01-26,2023-01-27,2023-01-30,Second Class,Delivered,CUST-02158,Pam Baker,Consumer,New James,Florida,South,PROD-025,3-Ring Binder Set,Office Supplies,Organization,15,5,1,0.0,15.0,10.0,66.67,Bank Transfer,1,4,0
1733,ORD-001293,2023-01-26,2023-01-27,2023-01-30,Second Class,Delivered,CUST-02542,Michelle Prince,Corporate,East Elizabeth,California,West,PROD-006,HP Spectre x360,Technology,Laptops,1099,800,1,0.0,1099.0,299.0,27.21,PayPal,1,4,0
1734,ORD-001293,2023-01-26,2023-01-27,2023-01-30,Second Class,Delivered,CUST-02542,Michelle Prince,Corporate,East Elizabeth,California,West,PROD-006,HP Spectre x360,Technology,Laptops,1099,800,1,0.0,1099.0,299.0,27.21,PayPal,1,4,0
2443,ORD-001830,2023-02-06,2023-02-07,2023-02-08,Same Day,Delivered,CUST-03514,Joshua Hopkins,Consumer,South Timothyton,California,West,PROD-027,Bamboo Desk Organizer,Office Supplies,Organization,25,10,1,0.0,25.0,15.0,60.00,Credit Card,1,2,0
2444,ORD-001830,2023-02-06,2023-02-07,2023-02-08,Same Day,Delivered,CUST-03514,Joshua Hopkins,Consumer,South Timothyton,California,West,PROD-027,Bamboo Desk Organizer,Office Supplies,Organization,25,10,1,0.0,25.0,15.0,60.00,Credit Card,1,2,0


These rows are identical in **every** column, including the `order_id`. The most likely explanation is that the same line was registered twice,
so I will remove the copies (if I kept them, the revenue of these orders would be counted twice).

In [8]:
rows_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)

print(f"Removed {rows_before - len(df)} rows. Rows now: {len(df):,}")

Removed 68 rows. Rows now: 80,915


There is another type of "duplicate" worth checking: the same product appearing twice in the same order.

In [9]:
same_product = df[df.duplicated(subset=["order_id", "product_id"], keep=False)]
print("Rows with the same order_id + product_id:", len(same_product))

same_product.sort_values("order_id")[["order_id", "product_name", "quantity", "discount", "sales"]].head(6)

Rows with the same order_id + product_id: 1574


,order_id,product_name,quantity,discount,sales
123,ORD-000098,Sticky Notes 12-Pack,2,0.25,12.0
124,ORD-000098,Sticky Notes 12-Pack,3,0.00,24.0
160,ORD-000122,Sticky Notes 12-Pack,2,0.05,15.2
161,ORD-000122,Sticky Notes 12-Pack,1,0.00,8.0
215,ORD-000164,Samsung Galaxy S24,3,0.10,2157.3
216,ORD-000164,Samsung Galaxy S24,2,0.15,1358.3


Here the quantity and discount are different between the two lines, so they look like two separate lines of the same order
(for example, part of the units bought with a promotion). These are **not** errors, so I keep them.

## 4. Consistency checks

The dataset has some calculated columns (`sales`, `profit`, `profit_margin`, `days_to_ship`, `days_to_deliver`).
I want to make sure they really match the formulas, otherwise any analysis on them would be wrong.

In [10]:
# sales should be: unit price x quantity x (1 - discount)
expected_sales = df["unit_price"] * df["quantity"] * (1 - df["discount"])
wrong_sales = (expected_sales - df["sales"]).abs() > 0.01

# profit should be: sales - cost of the units sold
expected_profit = df["sales"] - df["cost_price"] * df["quantity"]
wrong_profit = (expected_profit - df["profit"]).abs() > 0.01

# profit margin (%) should be: profit / sales x 100
expected_margin = df["profit"] / df["sales"] * 100
wrong_margin = (expected_margin - df["profit_margin"]).abs() > 0.01

print("Rows with wrong sales: ", wrong_sales.sum())
print("Rows with wrong profit:", wrong_profit.sum())
print("Rows with wrong margin:", wrong_margin.sum())

Rows with wrong sales:  0
Rows with wrong profit: 0
Rows with wrong margin: 0


In [11]:
# days_to_ship and days_to_deliver are both counted from the order date
wrong_ship_days = (df["ship_date"] - df["order_date"]).dt.days != df["days_to_ship"]
wrong_deliver_days = (df["deliver_date"] - df["order_date"]).dt.days != df["days_to_deliver"]

print("Rows with wrong days_to_ship:   ", wrong_ship_days.sum())
print("Rows with wrong days_to_deliver:", wrong_deliver_days.sum())

# dates must be in the right order: order -> ship -> deliver
print("Shipped before ordered:  ", (df["ship_date"] < df["order_date"]).sum())
print("Delivered before shipped:", (df["deliver_date"] < df["ship_date"]).sum())

Rows with wrong days_to_ship:    0
Rows with wrong days_to_deliver: 0
Shipped before ordered:   0
Delivered before shipped: 0


In [12]:
# order_status and is_returned should say the same thing
pd.crosstab(df["order_status"], df["is_returned"])

is_returned,0,1
order_status,,
Delivered,75438,0
Returned,0,5477


All the calculated columns are correct. Next: information that should be the same for the whole order or for the same customer.

In [13]:
# inside one order, these columns should have only one value
order_level_columns = ["customer_id", "order_date", "ship_mode", "payment_method", "order_status"]

for col in order_level_columns:
    orders_with_more_than_one_value = (df.groupby("order_id")[col].nunique() > 1).sum()
    print(f"{col:15} -> orders with more than one value: {orders_with_more_than_one_value}")

customer_id     -> orders with more than one value: 0
order_date      -> orders with more than one value: 0
ship_mode       -> orders with more than one value: 0
payment_method  -> orders with more than one value: 0
order_status    -> orders with more than one value: 2635


Everything matches except `order_status`: in some orders one product was returned and the other was not.
This makes sense, it means **returns happen per product, not per order**. I will keep that in mind for the return analysis.

In [14]:
# each customer should always have the same name, segment and location
for col in ["customer_name", "segment", "city", "state", "region"]:
    customers_with_more_than_one_value = (df.groupby("customer_id")[col].nunique() > 1).sum()
    print(f"{col:14} -> customers with more than one value: {customers_with_more_than_one_value}")

print()
print("Unique customer IDs:  ", df["customer_id"].nunique())
print("Unique customer names:", df["customer_name"].nunique())

customer_name  -> customers with more than one value: 0
segment        -> customers with more than one value: 0
city           -> customers with more than one value: 0
state          -> customers with more than one value: 0
region         -> customers with more than one value: 0

Unique customer IDs:   7962
Unique customer names: 7527


The customer information is consistent. There are fewer names than IDs because different customers can have the same name
(e.g. two "John Johnson"), so for any customer analysis I will always use `customer_id`, never the name.

In [15]:
# each state should belong to only one region, and each city to only one state
print("States in more than one region:", (df.groupby("state")["region"].nunique() > 1).sum())
print("Cities in more than one state: ", (df.groupby("city")["state"].nunique() > 1).sum())

df["city"].value_counts().head()

States in more than one region: 0
Cities in more than one state:  988


city
South John        111
East James         89
South Michael      87
West Stephanie     85
Port Michael       81
Name: count, dtype: int64

States and regions are fine, but almost a thousand city names appear in more than one state, and names like "South John" or "Lake Angela"
are clearly randomly generated. **The city column is not reliable**, so I will do the geographic analysis at state and region level only.

## 5. Categorical values and outliers

In [16]:
# check the text columns with few options for typos or strange values
categorical_columns = ["ship_mode", "order_status", "segment", "region", "category", "sub_category", "payment_method"]

for col in categorical_columns:
    print(f"{col}: {sorted(df[col].unique())}")

ship_mode: ['First Class', 'Same Day', 'Second Class', 'Standard Class']
order_status: ['Delivered', 'Returned']
segment: ['Consumer', 'Corporate', 'Home Office']
region: ['Midwest', 'Northeast', 'South', 'West']
category: ['Furniture', 'Office Supplies', 'Technology']
sub_category: ['Accessories', 'Chairs', 'Desks', 'Laptops', 'Organization', 'Paper', 'Phones', 'Storage', 'Writing']
payment_method: ['Bank Transfer', 'Credit Card', 'Debit Card', 'PayPal']


In [17]:
# each product should always have the same price, cost and category
product_info = df.groupby("product_id").agg(
    product_name=("product_name", "first"),
    category=("category", "first"),
    n_prices=("unit_price", "nunique"),
    n_costs=("cost_price", "nunique"),
    unit_price=("unit_price", "first"),
    cost_price=("cost_price", "first"),
)
print("Products with more than one price:", (product_info["n_prices"] > 1).sum())
print("Products with more than one cost: ", (product_info["n_costs"] > 1).sum())

product_info.sort_values("unit_price", ascending=False)[["product_name", "category", "unit_price", "cost_price"]]

Products with more than one price: 0
Products with more than one cost:  0


,product_name,category,unit_price,cost_price
product_id,,,,
PROD-004,MacBook Air M3,Technology,1299,950
PROD-005,Dell XPS 15,Technology,1199,870
PROD-006,HP Spectre x360,Technology,1099,800
PROD-001,iPhone 15,Technology,899,650
PROD-002,Samsung Galaxy S24,Technology,799,580
PROD-003,Google Pixel 8,Technology,699,490
PROD-013,Electric Standing Desk,Furniture,599,300
PROD-014,L-Shaped Corner Desk,Furniture,399,180
PROD-010,Ergonomic Office Chair,Furniture,349,180


No typos, and each product has a fixed price. There are 27 products, from 7 USD pens to a 1,299 USD MacBook. Now the numeric columns:

In [18]:
df[["unit_price", "cost_price", "quantity", "discount", "sales", "profit", "profit_margin", "days_to_deliver"]].describe().round(2)

,unit_price,cost_price,quantity,discount,sales,profit,profit_margin,days_to_deliver
count,80915.00,80915.00,80915.00,80915.00,80915.00,80915.00,80915.00,80915.00
mean,273.51,179.24,1.76,0.08,441.62,127.02,51.08,6.14
std,395.39,288.62,1.04,0.08,789.97,201.60,17.86,1.94
min,7.00,2.00,1.00,0.00,4.90,-203.50,-4.48,2.00
25%,12.00,4.00,1.00,0.00,16.00,10.00,39.76,4.00
50%,39.00,15.00,1.00,0.05,53.10,30.00,57.68,6.00
75%,349.00,180.00,2.00,0.15,597.00,174.05,64.91,8.00
max,1299.00,950.00,5.00,0.30,6495.00,1745.00,75.00,9.00


In [19]:
# outliers in sales using the IQR rule
q1 = df["sales"].quantile(0.25)
q3 = df["sales"].quantile(0.75)
iqr = q3 - q1
upper_limit = q3 + 1.5 * iqr

sales_outliers = df[df["sales"] > upper_limit]
print(f"Upper limit: {upper_limit:.2f} USD")
print(f"Lines above the limit: {len(sales_outliers):,} ({len(sales_outliers) / len(df):.1%})")
print()
print(sales_outliers["sub_category"].value_counts())

Upper limit: 1468.50 USD
Lines above the limit: 6,677 (8.3%)

sub_category
Laptops    3584
Phones     2575
Desks       449
Chairs       69
Name: count, dtype: int64


The IQR rule flags many lines as outliers, but they are all laptops and phones bought in more than one unit.
They are real (and very important!) sales, not errors, so I **don't remove them**. The prices are very different between categories,
so a "big" sale for paper is not the same as a big sale for laptops.

The **negative profits** (min = -203.5) also need a look:

In [20]:
loss_lines = df[df["profit"] < 0]
print(f"Lines sold at a loss: {len(loss_lines)}")

loss_lines.groupby(["product_name", "discount"]).size()

Lines sold at a loss: 305


product_name        discount
Dell XPS 15         0.3         54
Google Pixel 8      0.3         53
HP Spectre x360     0.3         53
MacBook Air M3      0.3         56
Samsung Galaxy S24  0.3         46
iPhone 15           0.3         43
dtype: int64

All the lines with a loss are phones and laptops sold with the maximum discount (30%). The product cost is high for these, so a 30% discount
puts the price below cost. Not a data error, but a business problem that I will look at in the EDA.

Last check - shipping times per shipping mode:

In [21]:
df.groupby("ship_mode")[["days_to_ship", "days_to_deliver"]].agg(["min", "mean", "max"]).round(2)

days_to_ship           days_to_deliver          
                        min  mean max             min  mean max
ship_mode                                                      
First Class               1  1.50   2               3  3.50   4
Same Day                  1  1.49   2               2  2.49   3
Second Class              1  1.50   2               4  5.00   6
Standard Class            1  1.50   2               6  7.49   9

**Interesting:** the "Same Day" option never ships on the same day - it takes 1 or 2 days to ship, exactly like the other options.
The difference between modes is only in the delivery time. This can be a data generation issue or a real problem
(customers paying for a service they don't get). I keep the data as it is, but I will mention it in the conclusions.

## 6. New columns

To make the time analysis easier I create some columns from the order date, and the total cost of each line.

In [22]:
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["order_quarter"] = df["order_date"].dt.quarter
df["year_month"] = df["order_date"].dt.to_period("M").astype(str)   # e.g. "2023-01"
df["weekday"] = df["order_date"].dt.day_name()

# total cost of the line (useful to calculate margins by group)
df["total_cost"] = df["cost_price"] * df["quantity"]

df[["order_date", "order_year", "order_month", "order_quarter", "year_month", "weekday", "total_cost"]].head()

,order_date,order_year,order_month,order_quarter,year_month,weekday,total_cost
0,2023-01-01,2023,1,1,2023-01,Sunday,80
1,2023-01-01,2023,1,1,2023-01,Sunday,2
2,2023-01-01,2023,1,1,2023-01,Sunday,2
3,2023-01-01,2023,1,1,2023-01,Sunday,490
4,2023-01-01,2023,1,1,2023-01,Sunday,580


## 7. Save the clean data

I save two things:
- **A clean CSV** -> used in the EDA notebook (`02_eda.ipynb`)
- **A SQLite database** -> used in the SQL notebook (`03_sql_analysis.ipynb`)

For the database, instead of one big table, I split the data into 4 tables like a real sales database would have.
This way the SQL part also shows JOINs between tables:

| Table | One row per | Columns |
|---|---|---|
| `customers` | customer | id, name, segment, city, state, region |
| `products` | product | id, name, category, sub-category, price, cost |
| `orders` | order | id, dates, ship mode, customer, payment method |
| `order_items` | product inside an order | order, product, quantity, discount, sales, profit, return flag |

In [23]:
df.to_csv("../data/processed/sales_clean.csv", index=False)
print("Clean CSV saved:", df.shape)

Clean CSV saved: (80915, 33)


In [24]:
customers = df[["customer_id", "customer_name", "segment", "city", "state", "region"]].drop_duplicates()

products = df[["product_id", "product_name", "category", "sub_category", "unit_price", "cost_price"]].drop_duplicates()

orders = df[["order_id", "order_date", "ship_date", "deliver_date", "ship_mode",
             "customer_id", "payment_method"]].drop_duplicates()

order_items = df[["order_id", "product_id", "quantity", "discount", "sales", "profit",
                  "order_status", "is_returned"]]

# the IDs must be unique in the first 3 tables, otherwise the JOINs would duplicate rows
print("customers:  ", len(customers), "rows | unique ids:", customers["customer_id"].nunique())
print("products:   ", len(products), "rows | unique ids:", products["product_id"].nunique())
print("orders:     ", len(orders), "rows | unique ids:", orders["order_id"].nunique())
print("order_items:", len(order_items), "rows")

customers:   7962 rows | unique ids: 7962
products:    27 rows | unique ids: 27
orders:      60000 rows | unique ids: 60000
order_items: 80915 rows


In [25]:
# SQLite has no date type, so I save the dates as text in the format YYYY-MM-DD
for col in ["order_date", "ship_date", "deliver_date"]:
    orders[col] = orders[col].dt.strftime("%Y-%m-%d")

conn = sqlite3.connect("../data/sales.db")

customers.to_sql("customers", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)
order_items.to_sql("order_items", conn, if_exists="replace", index=False)

# quick check: the total sales in the database must be the same as in the dataframe
db_total = pd.read_sql_query("SELECT SUM(sales) AS total FROM order_items", conn)["total"][0]
print(f"Total sales in the database: {db_total:,.2f}")
print(f"Total sales in the dataframe: {df['sales'].sum():,.2f}")

conn.close()

Total sales in the database: 35,733,463.75
Total sales in the dataframe: 35,733,463.75


## Summary

- The data was already in good shape: **no missing values** and all calculated columns are correct.
- Removed **68 exact duplicate rows**. Kept the lines with the same product in the same order, because they have different quantities/discounts.
- Converted the 3 date columns to datetime.
- Found some limitations to keep in mind:
  - **Returns are per product**, not per order.
  - **City names are not reliable** (random names, repeated in different states) -> geography only by state/region.
  - **"Same Day" shipping doesn't ship on the same day.**
  - A few hundred lines of phones/laptops **sell at a loss** when the discount is 30%.
- Created time columns (year, month, quarter, year_month, weekday) and `total_cost`.
- Saved a clean CSV and a SQLite database with 4 tables.

Next: [02_eda.ipynb](02_eda.ipynb)